In [25]:
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
from tqdm import tqdm

In [26]:
tqdm.pandas()

In [18]:
import pandas as pd
import json
from datasets import Dataset

In [39]:
data = pd.read_csv("../data/mimic3/mimic3_adhf_social_history_0406.csv", index_col = 0)

In [40]:
tokenizer = AutoTokenizer.from_pretrained("../output/roberta_fold_1_lr_5e-5/")
model = AutoModelForTokenClassification.from_pretrained("../output/roberta_fold_1_lr_5e-5/")

# The aggregation_strategy="simple" tells the pipeline to merge sub-words back together
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

Device set to use cuda:0


In [41]:
# 1. Convert the text paragraph into a list of sentences
# We use a lambda function to make sure we don't crash on empty or NaN values
data['social_history_sentences'] = data['social_history'].progress_apply(
    lambda text: sent_tokenize(text) if isinstance(text, str) else []
)

# 2. Explode the list into multiple rows
# This creates a new dataframe where every sentence gets its own row
df_exploded = data.explode('social_history_sentences')

# 3. Reset the index
# Explode keeps the original row index, so resetting it gives each new row a clean number
df_exploded = df_exploded.reset_index(drop=True)

100%|███████████████████████████████████| 3600/3600 [00:00<00:00, 21705.90it/s]


In [43]:
def get_predictions(sentence):
    # Handle any empty rows or NaNs safely
    if not isinstance(sentence, str) or sentence.strip() == "":
        return []
    
    # The pipeline returns a list of dictionaries with the entities
    return ner_pipeline(sentence)

# 4. Apply the prediction to your exploded dataframe
df_exploded['predicted_tags'] = df_exploded['social_history_sentences'].progress_apply(get_predictions)


100%|███████████████████████████████████| 17980/17980 [02:21<00:00, 126.64it/s]


In [44]:
def helper(predicted_list):
    tags_list = []
    for i in predicted_list:
        tags_list.append(i['entity_group'])
    return ','.join(tags_list)

In [46]:
df_exploded['tags'] = df_exploded['predicted_tags'].apply(helper)

In [50]:
df_exploded = df_exploded.drop('predicted_tags',axis=1)

In [52]:
df_exploded.to_csv("../data/mimic3/mimic3_adhf_social_history_0406_tags.csv")